# 04. Two-Stage Pipeline with Country Knowledge Base

This notebook supports:
- direct prompt editing,
- configurable generation parameters,
- automatic run logging,
- summary table for prompt/parameter comparison.


## Step 1: Install Dependencies


In [ ]:
!git clone https://github.com/alxefremov/esmt-workshop.git
%pip install -q -r requirements.txt


## Step 2: Load Shared Utilities


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Resolve repository root both for local runs and Google Colab.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path('/content/ESMT_Workshop')]:
    if (candidate / 'src' / 'esmt_workshop').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Project root not found. Run this notebook from the ESMT_Workshop repository.'
    )

# Make local package importable inside notebook execution context.
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report
from esmt_workshop.experiment_logging import load_experiment_history, log_experiment_run
from esmt_workshop.student_api import process_batch_addresses, process_single_address

print('PROJECT_ROOT =', PROJECT_ROOT)


## Step 3: Student Identity and Generation Controls


In [ ]:
from google.colab import auth

auth.authenticate_user()

gcloud_email = !gcloud config get-value account
print(f"Authenticated as: {gcloud_email[0] if gcloud_email else 'Unknown'}")
os.environ['WORKSHOP_EMAIL'] = gcloud_email[0] if gcloud_email else os.environ['WORKSHOP_EMAIL']

In [ ]:
# Notebook metadata used in experiment history logs.
NOTEBOOK_NAME = '04_two_stage_with_kb'

# Students only provide email; proxy endpoint details are managed by organizers.
STUDENT_EMAIL = os.getenv('WORKSHOP_EMAIL', 'student@example.com')

# Exposed generation controls available in every processing call.
TEMPERATURE = 0.0
TOP_P = 1.0
TOP_K = 40
MAX_TOKENS = 512
MAX_WORKERS = 8

# Free-text note for this run (optional, appears in history table).
RUN_NOTES = ''

STAGE_NAME = 'two_stage'
MODEL_NAME = 'gemini-2.5-pro'
COUNTRY_MODEL = 'gemini-2.5-flash'
USE_GUARDRAILS = False


## Step 4: Edit Prompt Directly in Notebook


In [ ]:
# Set to False to use built-in stage prompt logic.
USE_CUSTOM_PROMPT = True
PROMPT_LABEL = 'notebook_editable_prompt_v1'
PROMPT_TEMPLATE = '''You are an AML address parser.

Detected country:
{country}

Country format guidance:
{kb_text}

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
- Use country guidance only for mapping decisions.
- Do not invent missing tokens.
- Country Code must be ISO alpha-2 uppercase.
- Use empty strings when unknown.
'''

custom_prompt = PROMPT_TEMPLATE if USE_CUSTOM_PROMPT else None
print('Prompt length:', len(PROMPT_TEMPLATE) if custom_prompt else 0)


## Step 5: Run Pipeline


In [ ]:
import time

dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')
input_df = dev_df

# Execute configured stage and measure runtime.
t0 = time.perf_counter()

pred_df = process_batch_addresses(
    input_df,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    country_model=COUNTRY_MODEL if COUNTRY_MODEL else None,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=custom_prompt,
    kb_csv_path=str(PROJECT_ROOT / 'data/input/address_formats.csv'),
    max_workers=MAX_WORKERS,
)
runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))
pred_df.head(5)


## Step 6: Evaluate Output


In [ ]:
report = evaluate_predictions(pred_df, input_df, email=STUDENT_EMAIL)
print(report['summary'])
display(report['field_metrics'])


## Step 7: Save Artifacts and Log the Run


In [ ]:
out_dir = PROJECT_ROOT / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

pred_path = out_dir / '04_two_stage_with_kb_predictions.csv'
pred_df.to_csv(pred_path, index=False)

report_dir = out_dir / 'report_04_two_stage_with_kb'
if report.get('summary'):
    save_evaluation_report(report, report_dir)

run_record = log_experiment_run(
    output_root=PROJECT_ROOT / 'outputs',
    notebook_name=NOTEBOOK_NAME,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    country_model=COUNTRY_MODEL if COUNTRY_MODEL else None,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    max_workers=MAX_WORKERS,
    use_guardrails=USE_GUARDRAILS,
    kb_csv_path=str(PROJECT_ROOT / 'data/input/address_formats.csv'),
    prompt_template=custom_prompt,
    prompt_label=PROMPT_LABEL if USE_CUSTOM_PROMPT else 'stage_default_prompt',
    runtime_sec=runtime_sec,
    metrics_summary=report.get('summary', {}),
    notes=RUN_NOTES,
    predictions_path=pred_path,
    report_dir=report_dir if report.get('summary') else '',
)

print('Saved predictions:', pred_path)
if report.get('summary'):
    print('Saved report:', report_dir)
print('Logged run_id:', run_record['run_id'])


## Step 8: Prompt/Parameter History Summary


In [ ]:
history_df = load_experiment_history(output_root=PROJECT_ROOT / 'outputs', notebook_name=NOTEBOOK_NAME)
if history_df.empty:
    print('No runs logged yet.')
else:
    raw_cols = [
        'created_at_utc', 'stage', 'model', 'country_model', 'prompt_label',
        'temperature', 'top_p', 'top_k', 'micro_accuracy', 'runtime_sec',
        'row_exact_match', 'prompt_hash', 'notes'
    ]
    display(history_df[raw_cols].head(100))

    summary_df = (
        history_df.groupby(
            ['prompt_label', 'model', 'country_model', 'temperature', 'top_p', 'top_k', 'use_guardrails', 'mock_mode'],
            dropna=False,
        )
        .agg(
            runs=('run_id', 'count'),
            micro_accuracy_mean=('micro_accuracy', 'mean'),
            row_exact_match_mean=('row_exact_match', 'mean'),
            runtime_sec_mean=('runtime_sec', 'mean'),
        )
        .reset_index()
        .sort_values(['micro_accuracy_mean', 'row_exact_match_mean'], ascending=False)
    )

    display(summary_df.head(30))


## Assignment

1. Compare with advanced single-stage results.
2. Find countries where KB context helps most.
3. Continue with 05_input_guardrails.ipynb.
